In [1]:
# ch04_HUMAN_IN_THE_LOOP.ipynb

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langgraph.types import interrupt
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
import os

#llm = init_chat_model("openai:gpt-4.1")
llm = init_chat_model(
    model=f"google_genai:{os.getenv('GEMINI_MODEL')}",
    api_key=os.getenv('GEMINI_API_KEY'),
)

@tool
def human_assistance(query: str):
    """사람의 도움이 필요할 때 호출되는 도구입니다."""
    human_response = interrupt({"query": query})  # 실행 일시 중단
    return human_response

In [5]:
search_tool = TavilySearch(max_results=2)
tools = [search_tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

# 상태 정의
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 챗봇 노드 정의
def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    return {"messages": [message]}


In [8]:
# 그래프 초기화 및 메모리 설정
graph_builder = StateGraph(State)
memory = MemorySaver()

# 노드 및 엣지 추가
graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

# 그래프 컴파일
graph = graph_builder.compile(checkpointer=memory)

In [16]:
user_input = "AI 에이전트를 만드는 데 전문가의 조언이 필요해요. 사람의 도움을 받을 수 있을까요?"
config = {"configurable": {"thread_id": "1"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

AI 에이전트를 만드는 데 전문가의 조언이 필요해요. 사람의 도움을 받을 수 있을까요?
================================== Ai Message ==================================
Tool Calls:
  human_assistance (e4249721-b1b0-40c0-a7a7-b852caad0b1a)
 Call ID: e4249721-b1b0-40c0-a7a7-b852caad0b1a
  Args:
    query: AI 에이전트를 만드는 데 전문가의 조언이 필요해요.
================================== Ai Message ==================================
Tool Calls:
  human_assistance (e4249721-b1b0-40c0-a7a7-b852caad0b1a)
 Call ID: e4249721-b1b0-40c0-a7a7-b852caad0b1a
  Args:
    query: AI 에이전트를 만드는 데 전문가의 조언이 필요해요.


In [17]:
snapshot = graph.get_state(config)
print(snapshot.next)  # 출력: ('tools',)

('tools',)


In [18]:
from langgraph.types import Command

human_response = (
    "전문가입니다. LangGraph를 활용해 AI 에이전트를 만들어보세요. "
    "간단한 자율 에이전트보다 훨씬 더 확장성 있고 안정적입니다."
)

human_command = Command(resume={"data": human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_assistance (e4249721-b1b0-40c0-a7a7-b852caad0b1a)
 Call ID: e4249721-b1b0-40c0-a7a7-b852caad0b1a
  Args:
    query: AI 에이전트를 만드는 데 전문가의 조언이 필요해요.
================================= Tool Message =================================
Name: human_assistance

{"data": "전문가입니다. LangGraph를 활용해 AI 에이전트를 만들어보세요. 간단한 자율 에이전트보다 훨씬 더 확장성 있고 안정적입니다."}
================================== Ai Message ==================================

LangGraph를 활용해 AI 에이전트를 만들어보세요. 간단한 자율 에이전트보다 훨씬 더 확장성 있고 안정적입니다.
